In [0]:
# 1. Get the list of unique years from your SQL table as a list of strings
year_data = spark.sql("SELECT DISTINCT YEAR(order_in_date) as yr FROM my_project_db.gold_sales_fact ORDER BY yr DESC").collect()
years = [str(row['yr']) for row in year_data]

# 2. Create the dropdown using the dynamic list
dbutils.widgets.dropdown("selected_year", years[0], years)

# 3. Retrieve the selected value for use in subsequent SQL cells
selected_year = dbutils.widgets.get("selected_year")

In [0]:
%sql
WITH DelayCounts AS (
    SELECT 
        p.division,
        f.reason_name, 
        COUNT(*) as occurrence_count
    FROM my_project_db.gold_sales_fact f
    JOIN my_project_db.gold_dim_products p ON f.invt_id = p.invt_id
    -- Added the year filter here
    WHERE f.reason_name IS NOT NULL 
      AND YEAR(f.order_in_date) = :selected_year
    GROUP BY p.division, f.reason_name
),
RankedDelays AS (
    SELECT 
        division,
        reason_name,
        occurrence_count,
        ROW_NUMBER() OVER(PARTITION BY division ORDER BY occurrence_count DESC) as rank
    FROM DelayCounts
)
SELECT 
    division,
    reason_name as most_common_bottleneck,
    occurrence_count
FROM RankedDelays
WHERE rank = 1;